# Advanced 05 — Continuous & Adaptive Trust for Autonomous Agents

**Scenario:** a claims agent begins a valid long-running workflow. During execution, workload posture, behavior, credentials, relationships, and business risk change. Build a control plane that continuously re-evaluates authority and can reduce, step up, revoke, or quarantine the agent.

> CAEP/SSF sections model the standards; custom agent events use a private namespace and are intentionally not presented as standardized CAEP event types.


In [ ]:
from datetime import datetime,timedelta,timezone
from dataclasses import dataclass,field
import uuid,copy,hashlib,json
import pandas as pd
NOW=datetime.now(timezone.utc)
CAEP="https://schemas.openid.net/secevent/caep/event-type/"
PRIVATE="https://security.example/events/agent/"


## 1 — Continuous trust state

In [ ]:
@dataclass
class State:
    agent:str
    risk:dict=field(default_factory=lambda:{"identity":0,"workload":0,"behavior":0,"threat":0,"transaction":0})
    quarantined:bool=False
    revoked:bool=False
    tools:set=field(default_factory=lambda:{"knowledge.search","claim.read","claim.update","payment.request"})
state=State("agent:claims")


## 2 — Security event model

In [ ]:
def event(issuer,subject,event_type,details=None,ts=None):
    return {"iss":issuer,"jti":str(uuid.uuid4()),"iat":int(NOW.timestamp()),
            "subject":subject,"event_type":event_type,
            "event_timestamp":(ts or NOW).timestamp(),"details":details or {}}


## 3 — CAEP taxonomy

In [ ]:
caep_events={CAEP+x for x in [
"session-revoked","token-claims-change","credential-change","assurance-level-change",
"device-compliance-change","session-established","session-presented","risk-level-change"]}
len(caep_events)


## 4 — Private agent event taxonomy

In [ ]:
private_events={PRIVATE+x for x in [
"agent-quarantined","tool-anomaly","delegation-anomaly","release-withdrawn",
"evaluation-failed","cross-tenant-attempt"]}
private_events


## 5 — Transmitter trust

In [ ]:
trusted={"https://id.example","https://security.example","https://workload.example"}
e=event("https://security.example","agent:claims",PRIVATE+"tool-anomaly",{"severity":"high"})
assert e["iss"] in trusted


## 6 — Event integrity teaching model

In [ ]:
# Demonstration only: production SSF uses signed Security Event Tokens.
secret=b"demo-receiver-shared-secret"
def mac(e):
    return hashlib.sha256(secret+json.dumps(e,sort_keys=True).encode()).hexdigest()
signature=mac(e)
assert signature==mac(e)


## 7 — Subject correlation

In [ ]:
sessions={"session:1":{"agent":"agent:claims","workload":"spiffe://corp/prod/claims","tenant":"acme"}}
def affected_agent(subject):
    if subject.startswith("session:"): return sessions.get(subject,{}).get("agent")
    return subject
affected_agent("session:1")


## 8 — Duplicate detection

In [ ]:
seen=set()
def first_seen(evt):
    if evt["jti"] in seen:return False
    seen.add(evt["jti"]);return True
first_seen(e),first_seen(e)


## 9 — Event ordering

In [ ]:
latest={}
def ordered(evt):
    s=evt["subject"];t=evt["event_timestamp"]
    if t < latest.get(s,float("-inf")):return False
    latest[s]=t;return True
new=event("https://security.example","agent:claims",CAEP+"risk-level-change",ts=NOW)
old=event("https://security.example","agent:claims",CAEP+"risk-level-change",ts=NOW-timedelta(minutes=5))
ordered(new),ordered(old)


## 10 — Freshness

In [ ]:
def fresh(evt,max_age=timedelta(minutes=5),now=NOW):
    return now-datetime.fromtimestamp(evt["event_timestamp"],timezone.utc)<=max_age
fresh(new),fresh(old,timedelta(minutes=2))


## 11 — Normalize signal severity

In [ ]:
SEV={"low":10,"medium":35,"high":70,"critical":100}
def normalize(value):
    if isinstance(value,str):return SEV[value]
    if 0<=value<=1:return round(value*100)
    return max(0,min(100,int(value)))
[normalize(x) for x in ["low","high",.82,140]]


## 12 — Multidimensional risk

In [ ]:
state.risk["behavior"]=70
state.risk["workload"]=10
state.risk


## 13 — Risk aggregation

In [ ]:
def score(s):
    return max(s.risk.values())
score(state)


## 14 — Trust decay

In [ ]:
def decayed_positive_assurance(initial,age_minutes,half_life=30):
    return initial*(0.5**(age_minutes/half_life))
[round(decayed_positive_assurance(100,m),1) for m in [0,15,30,60,120]]


## 15 — Negative evidence precedence

In [ ]:
def decision(s):
    if s.quarantined:return "QUARANTINE"
    if s.revoked:return "REVOKE"
    r=score(s)
    if r>=90:return "QUARANTINE"
    if r>=70:return "STEP_UP"
    if r>=45:return "REDUCE"
    return "ALLOW"
decision(state)


## 16 — Adaptive actions

In [ ]:
for r in [10,50,75,95]:
    x=State("agent:test");x.risk["behavior"]=r
    print(r,decision(x))


## 17 — REDUCE tool authority

In [ ]:
def effective_tools(s):
    d=decision(s)
    if d=="ALLOW":return s.tools
    if d=="REDUCE":return s.tools & {"knowledge.search","claim.read"}
    return set()
x=State("agent:claims");x.risk["transaction"]=50
effective_tools(x)


## 18 — STEP-UP

In [ ]:
step_up_requirements={"claim.update":["fresh_workload_attestation"],
"payment.request":["human_approval","fresh_workload_attestation"]}
step_up_requirements["payment.request"]


## 19 — REVOKE scope

In [ ]:
revocations={"sessions":set(),"delegations":set(),"capabilities":set()}
revocations["sessions"].add("session:1")
revocations


## 20 — QUARANTINE

In [ ]:
q=State("agent:claims");q.quarantined=True
decision(q),effective_tools(q)


## 21 — Hysteresis

In [ ]:
def transition(current,risk):
    if current=="ALLOW" and risk>=60:return "REDUCE"
    if current=="REDUCE" and risk<40:return "ALLOW"
    return current
cur="ALLOW"
for r in [55,61,58,42,39]:
    cur=transition(cur,r);print(r,cur)


## 22 — Recovery conditions

In [ ]:
def can_recover(cooldown_done,fresh_attestation,incident_cleared,human_release):
    return all([cooldown_done,fresh_attestation,incident_cleared,human_release])
can_recover(True,True,True,False)


## 23 — Long-running action re-authorization

In [ ]:
planned={"action":"claim.update","authorized_at_plan_time":True}
state.risk["threat"]=100
planned["execute_now"]=decision(state)=="ALLOW"
planned


## 24 — Cache invalidation

In [ ]:
decision_cache={("agent:claims","claim.update"):"ALLOW"}
def invalidate_agent(agent):
    for k in list(decision_cache):
        if k[0]==agent:del decision_cache[k]
invalidate_agent("agent:claims")
decision_cache


## 25 — OPA-style adaptive input

In [ ]:
opa_input={"principal":"agent:claims","action":"claim.update",
"risk":{"score":score(state),"dimensions":state.risk},
"state":{"quarantined":state.quarantined,"revoked":state.revoked}}
opa_input


## 26 — Cedar forbid precedence

In [ ]:
def cedar_like(permit,quarantined,revoked,critical):
    if quarantined or revoked or critical:return "DENY"
    return "ALLOW" if permit else "DENY"
cedar_like(True,False,False,True)


## 27 — Relationship changes

In [ ]:
relationships={("alice","owns","agent:claims"),("agent:claims","assigned_to","claim:483")}
relationships.discard(("agent:claims","assigned_to","claim:483"))
("agent:claims","assigned_to","claim:483") in relationships


## 28 — Model CAEP session revocation

In [ ]:
caep_revoke=event("https://id.example","session:1",CAEP+"session-revoked",
{"reason_admin":"Policy violation"})
caep_revoke


## 29 — Model CAEP risk-level-change

In [ ]:
risk_change=event("https://security.example","agent:claims",CAEP+"risk-level-change",
{"current_level":"high","previous_level":"low"})
risk_change


## 30 — Stream outage policy

In [ ]:
def on_signal_stream_outage(action):
    return "STEP_UP" if action in {"claim.update","payment.request"} else "BOUNDED_STALE_OK"
[on_signal_stream_outage(a) for a in ["claim.read","claim.update"]]


## 31 — Signal poisoning

In [ ]:
fake=event("https://attacker.example","agent:claims",CAEP+"risk-level-change",{"current_level":"low"})
accepted=fake["iss"] in trusted
accepted


## 32 — Risk gaming

In [ ]:
# Keep detector internals out of model context; expose enforcement effect instead.
model_context={"authorization_mode":"READ_ONLY","reason_category":"security_state_changed"}
model_context


## 33 — Privacy minimization

In [ ]:
raw={"ip":"10.0.0.4","device_details":"sensitive","risk":82,"reason":"anomaly"}
policy_fact={"risk_level":"high","reason_category":"behavior_anomaly"}
policy_fact


## 34 — Decision evidence

In [ ]:
audit={"decision_id":str(uuid.uuid4()),"agent":"agent:claims",
"previous":"ALLOW","new":decision(state),"signals":[e["jti"]],
"policy_version":"adaptive-v12","risk_model":"risk-v4","time":NOW.isoformat()}
audit


## 35 — Enforcement latency

In [ ]:
event_time=NOW
received=NOW+timedelta(seconds=2)
evaluated=received+timedelta(seconds=1)
enforced=evaluated+timedelta(seconds=2)
(enforced-event_time).total_seconds()


## 36 — Shadow mode

In [ ]:
def shadow(current_enforced,adaptive_proposed):
    return {"enforced":current_enforced,"shadow":adaptive_proposed,
            "would_change":current_enforced!=adaptive_proposed}
shadow("ALLOW","REDUCE")


## 37 — Deterministic incident simulation

In [ ]:
timeline=[
("09:00","login","low"),
("09:05","normal tools","low"),
("09:17","workload non-compliant","medium"),
("09:19","tool anomaly","high"),
("09:20","threat match","critical")]
sim=State("agent:claims")
rows=[]
for t,name,sev in timeline:
    dim={"login":"identity","normal tools":"behavior","workload non-compliant":"workload",
         "tool anomaly":"behavior","threat match":"threat"}[name]
    sim.risk[dim]=SEV[sev]
    rows.append([t,name,score(sim),decision(sim)])
pd.DataFrame(rows,columns=["time","event","risk","decision"])


## 38 — Adversarial matrix

In [ ]:
attacks=[
"forged HIGH signal","forged LOW recovery","duplicate revocation","out-of-order recovery",
"event flood","stream outage","threshold flapping","queued write after quarantine",
"cached ALLOW after revoke","cross-tenant correlation","detector gaming","fake recovery"]
pd.DataFrame({"attack":attacks,"control":[
"transmitter trust","transmitter trust","idempotency","ordering",
"backpressure","fail policy","hysteresis","re-authorization",
"cache invalidation","compound subject","independent PEP","recovery policy"]})


# Capstone — Autonomous Agent Incident

Implement:

```text
Claims Agent starts valid session
       ↓
ALLOW claim.read + claim.update
       ↓
workload posture degrades
       ↓
REDUCE → read-only
       ↓
tool anomaly + HIGH risk
       ↓
STEP-UP
       ↓
critical threat signal
       ↓
QUARANTINE
       ↓
invalidate cached ALLOWs
revoke session/delegation
freeze queued writes
       ↓
fresh attestation + incident clear + human release
       ↓
controlled recovery
```

Requirements:

1. authenticate signal transmitters;
2. correlate events to the correct agent/session/workload/tenant;
3. deduplicate events;
4. reject stale/out-of-order state regression;
5. maintain multidimensional risk;
6. prioritize negative evidence;
7. use proportional actions;
8. implement hysteresis;
9. re-authorize queued sensitive actions;
10. invalidate authorization caches;
11. handle stream outage explicitly;
12. prevent signal poisoning;
13. keep raw telemetry out of model context;
14. record decision transitions and evidence;
15. measure event-to-enforcement latency;
16. require controlled recovery from quarantine.


# Review questions

1. Why is token validity insufficient for long-running agents?
2. How do continuous authentication and continuous authorization differ?
3. What does SSF standardize?
4. What is a Security Event Token?
5. Which event types are defined by CAEP 1.0?
6. Why should custom agent events use a private namespace?
7. Why must signal transmitters be authenticated?
8. How do duplicate and out-of-order events affect authorization?
9. Why separate event time from receipt time?
10. What is trust decay?
11. Why should negative evidence have explicit precedence?
12. When is REDUCE preferable to REVOKE?
13. What does STEP-UP mean for a non-human agent?
14. What problem does hysteresis solve?
15. Why must queued actions be re-authorized?
16. How can stale ALLOW caches defeat continuous access evaluation?
17. What should happen during a Shared Signals outage?
18. How can continuous telemetry create privacy risk?
19. What is signal poisoning?
20. Which latency should a continuous-access SLO measure?
